In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset

In [ ]:
class BUSISegmentationDataset(Dataset):

    def __init__(self, root_dir, image_size=256):
        self.root_dir = root_dir
        self.image_size = image_size
        self.samples = []

        classes = ["benign", "malignant", "normal"]

        for cls in classes:
            cls_dir = os.path.join(root_dir, cls)
            files = sorted(os.listdir(cls_dir))

            for f in files:
                if "_mask" in f:
                    continue

                img_path = os.path.join(cls_dir, f)
                mask_name = os.path.splitext(f)[0] + "_mask.png"
                mask_path = os.path.join(cls_dir, mask_name)

                if os.path.exists(mask_path):
                    self.samples.append((img_path, mask_path))
                else:
                    self.samples.append((img_path, None))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.image_size, self.image_size))
        image = image.astype(np.float32) / 255.0

        if mask_path is not None:
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(
                mask,
                (self.image_size, self.image_size),
                interpolation=cv2.INTER_NEAREST
            )
            mask = (mask > 127).astype(np.float32)
        else:
            mask = np.zeros(
                (self.image_size, self.image_size),
                dtype=np.float32
            )

        image = torch.tensor(image).permute(2, 0, 1)
        mask = torch.tensor(mask).unsqueeze(0)

        return image, mask

In [ ]:
ROOT_DIR = "/kaggle/input/datasets/sabahesaraki/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT/"

In [ ]:
dataset = BUSISegmentationDataset(ROOT_DIR, image_size=256)
print("Total samples:", len(dataset))

image, mask = dataset[0]
print(image.shape)
print(mask.shape)

In [ ]:
import matplotlib.pyplot as plt

image, mask = dataset[0]

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5),
    constrained_layout=True
)

axes[0].imshow(image.permute(1, 2, 0))
axes[0].set_title("Image")
axes[0].axis("off")

axes[1].imshow(mask.squeeze(), cmap="gray")
axes[1].set_title("Mask")
axes[1].axis("off")

plt.show()

In [ ]:
from torch.utils.data import random_split

total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)

val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

In [ ]:
def dice_score(pred, target, smooth=1e-6):

    pred = torch.sigmoid(pred)

    pred = (pred > 0.5).float()

    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()

    dice = (
        2.0 * intersection + smooth
    ) / (
        pred.sum() + target.sum() + smooth
    )

    return dice.item()

In [ ]:
def iou_score(pred, target, smooth=1e-6):

    pred = torch.sigmoid(pred)

    pred = (pred > 0.5).float()

    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()

    union = (
        pred.sum()
        + target.sum()
        - intersection
    )

    iou = (
        intersection + smooth
    ) / (
        union + smooth
    )

    return iou.item()

In [ ]:
class DiceLoss(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, pred, target):

        pred = torch.sigmoid(pred)

        pred = pred.view(-1)
        target = target.view(-1)

        intersection = (
            pred * target
        ).sum()

        dice = (
            2 * intersection + 1
        ) / (
            pred.sum()
            + target.sum()
            + 1
        )

        return 1 - dice

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()

dice_loss = DiceLoss()

def criterion(pred, target):

    bce = bce_loss(pred, target)

    dice = dice_loss(pred, target)

    return bce + dice

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [ ]:
def center_crop(skip, target):
    _, _, h, w = target.shape

    dh = skip.size(2) - h
    dw = skip.size(3) - w

    return skip[
        :,
        :,
        dh // 2 : dh // 2 + h,
        dw // 2 : dw // 2 + w
    ]

In [ ]:
class OriginalUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(2)

        # Encoder
        self.enc1 = DoubleConv(3, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder
        self.up4 = nn.ConvTranspose2d(
            1024, 512,
            kernel_size=2,
            stride=2
        )
        self.dec4 = DoubleConv(1024, 512)

        self.up3 = nn.ConvTranspose2d(
            512, 256,
            kernel_size=2,
            stride=2
        )
        self.dec3 = DoubleConv(512, 256)

        self.up2 = nn.ConvTranspose2d(
            256, 128,
            kernel_size=2,
            stride=2
        )
        self.dec2 = DoubleConv(256, 128)

        self.up1 = nn.ConvTranspose2d(
            128, 64,
            kernel_size=2,
            stride=2
        )
        self.dec1 = DoubleConv(128, 64)

        self.final = nn.Conv2d(
            64, 1,
            kernel_size=1
        )

    def forward(self, x, return_features=False):
        e1 = self.enc1(x)
        p1 = self.pool(e1)

        e2 = self.enc2(p1)
        p2 = self.pool(e2)

        e3 = self.enc3(p2)
        p3 = self.pool(e3)

        e4 = self.enc4(p3)
        p4 = self.pool(e4)

        b = self.bottleneck(p4)

        d4 = self.up4(b)
        e4_crop = center_crop(e4, d4)
        d4 = self.dec4(torch.cat([e4_crop, d4], dim=1))

        d3 = self.up3(d4)
        e3_crop = center_crop(e3, d3)
        d3 = self.dec3(torch.cat([e3_crop, d3], dim=1))

        d2 = self.up2(d3)
        e2_crop = center_crop(e2, d2)
        d2 = self.dec2(torch.cat([e2_crop, d2], dim=1))

        d1 = self.up1(d2)
        e1_crop = center_crop(e1, d1)
        d1 = self.dec1(torch.cat([e1_crop, d1], dim=1))

        out = self.final(d1)

        if return_features:
            features = {
                "e1": e1,
                "e2": e2,
                "e3": e3,
                "e4": e4,
                "b": b,
                "d4": d4,
                "d3": d3,
                "d2": d2,
                "d1": d1
            }
            return out, features

        return out

In [ ]:
import matplotlib.pyplot as plt
import math

In [ ]:
def visualize_feature_map(
        feature,
        title,
        n_channels=16):

    feature = feature[0].detach().cpu()

    n_channels = min(
        n_channels,
        feature.shape[0]
    )

    cols = 4
    rows = math.ceil(
        n_channels / cols
    )

    plt.figure(figsize=(12, 8))

    for i in range(n_channels):

        plt.subplot(
            rows,
            cols,
            i + 1
        )

        plt.imshow(
            feature[i],
            cmap="viridis"
        )

        plt.axis("off")

        plt.title(
            f"C{i}",
            fontsize=8,
            pad=1
        )

    plt.suptitle(
        title,
        fontsize=14,
        y=0.98
    )

    plt.subplots_adjust(
        left=0.01,
        right=0.99,
        bottom=0.01,
        top=0.92,
        wspace=0.02,
        hspace=0.02
    )

    plt.show()

In [ ]:
def show_all_features(features):

    visualize_feature_map(
        features["e1"],
        "Encoder 1"
    )

    visualize_feature_map(
        features["e2"],
        "Encoder 2"
    )

    visualize_feature_map(
        features["e3"],
        "Encoder 3"
    )

    visualize_feature_map(
        features["e4"],
        "Encoder 4"
    )

    visualize_feature_map(
        features["b"],
        "Bottleneck"
    )

    visualize_feature_map(
        features["d4"],
        "Decoder 4"
    )

    visualize_feature_map(
        features["d3"],
        "Decoder 3"
    )

    visualize_feature_map(
        features["d2"],
        "Decoder 2"
    )

    visualize_feature_map(
        features["d1"],
        "Decoder 1"
    )

In [ ]:
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = OriginalUNet().to(DEVICE)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
def evaluate(model, loader):

    model.eval()

    total_dice = 0
    total_iou = 0

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(DEVICE)
            masks = masks.to(DEVICE)

            outputs = model(images)

            total_dice += dice_score(
                outputs,
                masks
            )

            total_iou += iou_score(
                outputs,
                masks
            )

    avg_dice = (
        total_dice
        / len(loader)
    )

    avg_iou = (
        total_iou
        / len(loader)
    )

    return avg_dice, avg_iou

In [ ]:
from tqdm import tqdm

In [ ]:
NUM_EPOCHS = 50

for epoch in range(NUM_EPOCHS):

    model.train()

    train_loss = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"
    )

    for batch_idx, (images, masks) in enumerate(train_bar):

        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            masks
        )

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        train_bar.set_postfix(
            Loss=f"{loss.item():.4f}"
        )

    avg_loss = (
        train_loss
        / len(train_loader)
    )

    val_dice, val_iou = evaluate(
        model,
        val_loader
    )

    print(
        f"\nEpoch {epoch+1}"
        f"\nLoss : {avg_loss:.4f}"
        f"\nDice : {val_dice:.4f}"
        f"\nIoU  : {val_iou:.4f}"
    )

    if epoch % 5 == 0:

        model.eval()

        with torch.no_grad():

            sample_img, _ = next(
                iter(val_loader)
            )

            sample_img = sample_img[:1].to(
                DEVICE
            )

            pred, features = model(
                sample_img,
                return_features=True
            )

            print(
                "\nFeature Shapes:"
            )

            for k, v in features.items():

                print(
                    f"{k}: {v.shape}"
                )

            show_all_features(
                features
            )